# JN-N — The label as a data product

**What this is.** Every building in the berkeleybuild.com flyovers carries a small panel: address,
how many homes, how tall, who is building it, who owns the land. This notebook is where those
panels are designed, and where the editorial decision behind them — *which* six facts, out of
everything v2 knows — is made explicit instead of implicit.

**Why a notebook and not a script.** `gen_svg_labels.svg()` returns a finished label in **1.3 ms**.
The 5–30 minute wait that has governed every label decision so far is the *rasteriser*
(`qlmanage` at 0.53 s per label, `cairosvg` at 0.31 s), not the design. Here you change a line and
see it immediately.

**Derived, not hardcoded.** Every figure below is computed from `v_projects_flat` at run time and
gated against an external timestamped baseline
(`data/baselines/label_fields_baseline_*.json`). A legitimate change to the data means
**appending a new baseline**, never editing the logic to match.

| | |
|---|---|
| v2 sha at build | `31a4423bff5b4697` |
| baseline gated against | `label_fields_baseline_2026-09-04.json` |
| projects in scope | 895 (merged rows excluded) |

## §1 — One label, end to end

**Assumption.** A label is a pure function of one row of `v_projects_flat`. Nothing about the tour,
the camera or the geometry enters into it.

**Plan.** Pull the row for 2128 Oxford St, show the six lines `lines_for()` produces, and render the
SVG inline.

**Why this project.** Its numbers moved today. A gated write took it from the 2023 SB330 application
figures (485 units, 26 storeys) to the ZAB-approved plan set (456 units, 27 storeys). If the label
below says 456, the pipeline is tracking corrections rather than serving a cached rendering of a
superseded figure — which it was doing until this morning, because the PNG cache keyed on filename.

In [ ]:
import sys, sqlite3, hashlib, json, glob, os
sys.path.insert(0, os.path.expanduser('~/berkeley-data/scripts'))
import gen_svg_labels as G
from IPython.display import SVG, display

V2 = os.path.expanduser('~/berkeley-data/databases/berkeley_housing_v2.db')
def ro(p): return sqlite3.connect(f'file:{p}?mode=ro', uri=True)

os.chdir(os.path.expanduser('~/berkeley-data'))
ROWS = G.rows(False)          # the pipeline's own row source: adds uc_project, drops merged rows
BY = {str(r['address_display']): r for r in ROWS}
r = BY['2128 Oxford St']

for line in [x for x in G.lines_for(r) if x]:
    print(' ', line)
display(SVG(G.svg(r)))

**Found.** Six lines, and the SVG rendered in about a millisecond. The panel colour is the project's
status — orange for *Entitled* — which is the same encoding the flyover uses for the building
footprint, so the label and the building agree without the viewer being told the rule.

**Verify.** The units line should read **456**, not 485. If it reads 485 you are running against a
v2 that predates the 2026-09-04 write, and the gate in §5 will say so.

## §2 — What the data can actually support

**Assumption — and it is the one worth testing.** The label design assumes six lines are available.
They are not. `architect`, `developer` and `owner_current` are sparse; `height_feet` is sparser
still. A label design that assumes rich data produces mostly-empty boxes.

**Plan.** Count, for every field a label line needs, how many of the projects in scope actually
carry it. Then count how many of the six display lines each project can fill.

In [ ]:
LINE_FIELDS = {
    "address": "address_display",
    "units": "total_units",
    "status": "status_label",
    "storeys": "height_stories",
    "height": "height_feet",
    "filed": "filed_date",
    "architect": "architect",
    "developer": "developer",
    "owner": "owner_current"
}

n = len(ROWS)
print(f'{n} projects in scope\n')
cov = {}
for name, col in LINE_FIELDS.items():
    k = sum(1 for x in ROWS
            if x[col] is not None and str(x[col]).strip() not in ('', '0', 'None'))
    cov[name] = k
    print(f'  {name:10} {col:18} {k:5}  {100*k/n:5.1f}%')

hist = {}
for x in ROWS:
    hist[len([y for y in G.lines_for(x) if y])] = hist.get(len([y for y in G.lines_for(x) if y]), 0) + 1
print('\n  lines a project can fill -> how many projects')
for k in sorted(hist):
    print(f'    {k} lines  {hist[k]:5}   {"#" * int(60*hist[k]/n)}')

**Found.** The address and status are near-universal; the people are not. Of 895
projects, **852 carry an owner** and **50 an
architect**. The owner figure is high only because of the assessor join done on 2026-09-02 — before
that it was 28 out of 895, and the site was serving *that* number until today.

**Verify — and this is the point of the section.** Look at the histogram. The modal project fills
far fewer than six lines. **Designing a six-line label for a pipeline whose typical project supports
three is a design error, not a data problem.** The label happens to degrade gracefully (missing
lines are simply omitted and the box shrinks to fit), but the *design* was chosen against the
richest projects — the towers we orbit — and those are the exception.

## §3 — Choose your own label

**Assumption.** Which six facts appear is an editorial choice, not a technical constraint.

**Plan.** `lines_for()` is one function. Write a different one and the whole pipeline — SVG, PNG,
KMZ, flyover — follows. Below are three alternative labels over the same project: the shipped one,
an affordability-first one, and a money one.

**Why it matters.** This is the handle a reader gets. The video shows there is information on every
building; this cell is where they change what that information *is*.

In [ ]:
def label_affordability(r):
    t = r['total_units'] or 0
    aff = (r['eli_units'] or 0) + (r['vli_units'] or 0) + (r['li_units'] or 0) + (r['mod_units'] or 0)
    return [str(r['address_display']),
            f"{aff:,} affordable of {t:,}" if t else 'units unknown',
            f"{100*aff/t:.0f}% below market" if t else '',
            str(r['status_label'] or '')]

def label_money(r):
    av = r['assessed_value']; tax = r['est_annual_tax']
    return [str(r['address_display']),
            f"assessed ${av:,.0f}" if av else 'not yet assessed',
            f"tax ${tax:,.0f}/yr" if tax else '',
            f"{r['total_units'] or 0:,} units · {r['status_label']}"]

for name, fn in (('shipped', G.lines_for),
                 ('affordability', label_affordability),
                 ('money', label_money)):
    print(f'  --- {name} ---')
    for line in [x for x in fn(r) if x]:
        print('   ', line)
    print()

**Found.** Same building, three different arguments about it. The shipped label answers *what is
being built*; the affordability label answers *who gets to live there*; the money label answers
*what it is worth to the city*.

**What this could mislead about.** The affordability label divides by `total_units`, and for
2128 Oxford that denominator moved by 29 today. A percentage is only as stable as the number under
it — and unlike a raw count, a percentage hides that it moved. The money label is worse: an
`assessed_value` of $0 on a finished building usually means **reassessment lag, not zero value**,
so a money label would confidently print "not yet assessed" on a completed occupied tower. Neither
is wrong; both need the caveat the video has no room for. That asymmetry — the notebook can
qualify, the label cannot — is the honest reason the flyover is a teaser and not the analysis.

## §4 — The renderer that was quietly changing the design

**Assumption we held until today.** The panel was translucent (`fill-opacity="0.86"`), so buildings
showed faintly through it.

**Found — it never was.** `qlmanage` is macOS Quick Look, and it *flattens alpha against a light
background*. It rendered `#0d1117` at 0.86 as an opaque **(47,51,56)** grey. Every label in every
video shipped before 2026-09-04 is that grey. `cairosvg` renders the same SVG as **(13,17,23)**.

The design intent never reached a single frame, and nothing reported an error — the rasteriser
simply returned a different picture than the one specified.

In [ ]:
# Derived, not asserted: read the panel pixel straight out of a rendered label.
from pathlib import Path
try:
    from PIL import Image
    p = Path(os.path.expanduser('~/berkeley-data/scratch/2026-08-31/svg-labels/2128-oxford-st.png'))
    if p.exists():
        im = Image.open(p).convert('RGBA')
        print(f'  shipped label {p.name}: size {im.size}  panel pixel {im.load()[300,100]}')
        print('  (13,17,23,255) = cairosvg, faithful   (47,51,56,255) = qlmanage, flattened')
    else:
        print('  no rendered label on disk yet — run scripts/gen_svg_labels.py')
except ImportError:
    print('  PIL not available in this kernel; skipping the pixel check')

**Verify.** A faithful render reads `(13,17,23,255)`. Anything near `(47,51,56)` means the label was
made by Quick Look and is lighter than designed.

**The general lesson, which is not about labels.** A tool that silently returns *something
plausible* instead of what you asked for is worse than one that fails, because the output looks
fine. We only caught it by diffing two renderers pixel by pixel — by eye they were identical.

## §5 — The gate

**Assumption.** Coverage counts and the exact text of the anchor labels should not move unless v2
moved, and if they do move the notebook must say so rather than quietly render something new.

**Plan.** Recompute, compare against the newest timestamped baseline, and halt with a diagnosis on
mismatch. A legitimate change is an **appended** baseline, never an edit here.

In [ ]:
import glob
BG = os.path.expanduser('~/berkeley-data/data/baselines/label_fields_baseline_*.json')
hits = sorted(glob.glob(BG))
if not hits:
    print('  NO BASELINE — run: python scripts/v4/build_jn_n.py --write-baseline')
else:
    b = json.load(open(hits[-1]))
    print(f'  baseline {os.path.basename(hits[-1])}  (v2_sha then {b.get("v2_sha")})')
    bad = []
    if len(ROWS) != b['n_projects']:
        bad.append(f"n_projects: now {len(ROWS)}, baseline {b['n_projects']}")
    for name, k in cov.items():
        if k != b['coverage'].get(name):
            bad.append(f"coverage[{name}]: now {k}, baseline {b['coverage'].get(name)}")
    for a, bv in b['anchors'].items():
        rr = BY.get(a)
        now = [x for x in G.lines_for(rr) if x] if rr is not None else None
        s = hashlib.sha256('\n'.join(now).encode()).hexdigest()[:16] if now else None
        if s != (bv or {}).get('sha'):
            bad.append(f"anchor[{a}]\n      now      : {now}\n      baseline : {(bv or {}).get('lines')}")
    if bad:
        print('\n  GATE FAILED:')
        for x in bad: print('    -', x)
        print('\n  If the new value is CORRECT, append a new baseline. Never edit the logic to match.')
    else:
        print('  GATE PASSED — coverage and every anchor label match the baseline.')

**Found at build time.** Gate PASSED against
`label_fields_baseline_2026-09-04.json`.

**Verify — prove the gate can fail.** Edit `lines_for()` in `scripts/gen_svg_labels.py` (add a word
to the address line), re-run the cell above, and confirm it FAILS and prints the old and new text.
A gate that has never been seen to fail is not evidence of anything. This one was proven both ways;
so was the deploy gate it is modelled on.

## §6 — Coverage, drawn

**What this shows.** For each label line, the share of projects that can fill it. Read it as *how
often this line appears at all*, not how often it is interesting.

In [ ]:
import plotly.graph_objects as go
names = list(cov.keys()); vals = [100*cov[k]/len(ROWS) for k in names]
order = sorted(range(len(names)), key=lambda i: -vals[i])
fig = go.Figure(go.Bar(x=[vals[i] for i in order], y=[names[i] for i in order],
                       orientation='h', marker_color='#ff8000',
                       text=[f'{vals[i]:.0f}%  ({cov[names[i]]})' for i in order],
                       textposition='outside'))
fig.update_layout(title=f'Label line coverage across {len(ROWS)} projects',
                  xaxis_title='% of projects that can fill this line',
                  xaxis_range=[0,115], height=420,
                  paper_bgcolor='white', plot_bgcolor='white')
fig.show()

**What this could mislead about.** Coverage is not quality. `owner_current` shows high because every
Berkeley parcel has an assessor owner of record — but an owner of record is often a single-purpose
LLC that tells you nothing about who is actually behind a project. A bar at 95% invites you to read
"we know who owns these"; what it means is "we know what name is on the deed". The chart cannot
carry that distinction and the label carries it even less.

## §7 — Where a label comes from

```mermaid
graph LR
  A[CPRA permits xlsx] --> V2[(berkeley_housing_v2.db)]
  B[Alameda assessor<br/>berkeley.db parcels] --> V2
  C[Plan sets · 1.E tabulation forms<br/>ZAB packets] --> V2
  V2 --> VF[v_projects_flat]
  VF --> LF["gen_svg_labels.lines_for()<br/><b>the editorial choice</b>"]
  LF --> SVG["svg() — 1.3 ms"]
  SVG --> R{rasteriser}
  R -->|cairosvg 0.31s<br/>faithful| PNG[label PNG]
  R -->|qlmanage 0.53s<br/>FLATTENS ALPHA| PNG
  PNG --> KMZ["svg_label_tour.py<br/>package + gx:AnimatedUpdate"]
  KMZ --> GE[Google Earth]
  GE -->|Movie Maker, by hand| MP4[YouTube]
  VF --> EXP[export_explorer_data_v2.py] --> SITE[berkeleybuild.com explorer]
```

**Read it as:** one row of `v_projects_flat` feeds both the label and the website, so a gated write
moves both — but only if the caches downstream invalidate. Two did not, and both were found on
2026-09-04: the label PNG cache keyed on filename (fixed: it now hashes the SVG), and the explorer's
served data file needed a manual copy (fixed: the deploy gate now blocks a stale one).

**The diamond is the part to remember.** The rasteriser is the one step in this chain that can
change what the reader sees without changing any data, any code, or any figure in this notebook.

## §8 — What this notebook does not do

**It does not make the labels legible in video.** Measured from the 2026-09-04 Shattuck recording, a
pass-by label renders about **124 × 52 px on a 1920-wide frame** — an unreadable smudge. Only the
four orbited buildings are readable. Inside Earth you can pause and lean in; in a video nobody can.
That is not fixable here, because Earth ties icon size to camera distance.

**The route out is a different pipeline.** Google Earth Pro is scriptable — `GetViewInfo`,
`SetViewInfo`, `SaveScreenShot`, `GetStreamingProgress`, verified 2026-09-04 — with no Movie Maker
command. A notebook that steps the camera and captures frames knows exactly where the camera is,
so it can composite labels in **screen space** with a minimum legible size. That would dissolve the
placement problems this project has fought all along — the label sinking into the building, the
roof-versus-view-axis height, the radial pump around an elongated footprint — because none of them
exist in 2D.

Untested and honest about it: `SaveScreenShot` and `GetStreamingProgress` have not been exercised,
the projection needs calibrating against a known frame, and 470 seconds at 29.97 fps is ~14,000
scripted round-trips through Rosetta. That is the next spike, not a promise.